# Physicochemical reconstruction of atom contributions

**Kind:** experiment. This notebook can run independently of training.

Fit the primary seven-descriptor Ridge model on all 23,029 MMPs using the
bundled R fold-0 AC table, or set `ATOM_CONTRIBUTIONS_PATH` to a newly trained
model's `atom_contributions.csv`. Reconstruction is in-sample on both endpoints;
it does not measure generalization to unseen atoms or causal effects.


In [ ]:
LOCAL_PROJECT_ROOT = "."
# For a newly trained R fold 0, use notebooks_for_sglt2/output/sglt2/R/fold_0/atom_contributions.csv.
ATOM_CONTRIBUTIONS_PATH = "sglt2_data/reference_R_ac.csv.gz"
RIDGE_ALPHA = 1.0

## 1. Runtime and reference inputs

In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display

# Use the declared repository root in both Colab and local execution.
try:
    import google.colab
except ModuleNotFoundError as exc:
    if exc.name not in {"google", "google.colab"}:
        raise
    IS_COLAB = False
else:
    IS_COLAB = True
ROOT = Path(os.environ.get("XAI_CHEM_ROOT", LOCAL_PROJECT_ROOT)).expanduser().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print({"runtime": "colab" if IS_COLAB else "local", "root": str(ROOT), "python": sys.executable})

# Numerical tables, chemistry figures, and explicit CPU execution.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)
%matplotlib inline


## 2. Bind the seven raw descriptors and the selected teacher

In [ ]:
from xai_chem.reconstruction import load_mmp_inputs, fit_all_mmp_pairs

DESCRIPTORS = (
    "crest_boltzmann_weighted_displacement_A", "hessian_rmsf_A",
    "fukui_plus_raw", "fukui_minus_raw", "neutral_charge_square",
    "epsilon_kcal_mol", "sigma_angstrom",
)
teacher_path = ROOT / ATOM_CONTRIBUTIONS_PATH

mmp_rows, atom_rows = load_mmp_inputs(
    mmp_path=ROOT / "sglt2_data/sglt2_reconstruction_mmp.csv.gz",
    descriptor_path=ROOT / "sglt2_data/sglt2_descriptors.csv.gz",
    dataset_path=ROOT / "sglt2_data/sglt2.csv",
    teacher_path=teacher_path, teacher_fold=None, descriptor_columns=DESCRIPTORS,
)
display(atom_rows[["graph_id", "atom_idx", *DESCRIPTORS, "atom_contribution"]].head())
print({"teacher_path": str(teacher_path), "pairs": len(mmp_rows), "atoms": len(atom_rows), "ridge_alpha": RIDGE_ALPHA})

## 3. Pairwise standardization, fit, and exact effects

For each pair, fit the descriptors and atom labels of its A/B changed-region atoms:

$$
z_{id}=\frac{x_{id}-\mu_{d,AB}}{s_{d,AB}},\qquad
\tilde c_i=\frac{c_i-\bar c_{AB}}{s_{c,AB}}.
$$

`fit_pair` solves Ridge with `fit_intercept=False` in this standardized space:

$$
\hat\beta=\mathop{\arg\min}_{\beta}\left\lVert\tilde c-Z\beta\right\rVert_2^2+\alpha\lVert\beta\rVert_2^2,
\qquad e_{id}=s_{c,AB}z_{id}\hat\beta_d.
$$

The endpoint difference contains both the exact descriptor effects and the
intercept-sum term $(n_B-n_A)\bar c_{AB}$. Atom-count difference is not a fitted descriptor.
Zero descriptor variance uses StandardScaler's unit scale; constant labels use
the existing unit-scale convention. No augmentation is used.

In [ ]:
variants = {"ridge": {"model_kind": "ridge", "alpha": RIDGE_ALPHA, "augmentation_multiplier": 0}}
all_fit = fit_all_mmp_pairs(mmp_rows, atom_rows, variants=variants, descriptors=DESCRIPTORS)
display(all_fit.audit.head())
display(all_fit.fit_metrics.head())
display(all_fit.coefficients.head(14))
display(all_fit.concept_deltas.head(14))

## 4. Audit coverage and exact reconstruction

In [ ]:
metrics = all_fit.fit_metrics.set_index("pair_id")
effects = all_fit.concept_deltas.groupby("pair_id")["delta_effect"].sum()
reconstruction_gap = effects.loc[metrics.index].to_numpy() - metrics["fitted_delta_B_minus_A"].to_numpy()
print({"audited_pairs": len(all_fit.audit), "fitted_pairs": len(metrics),
       "max_exact_effect_gap": float(np.max(np.abs(reconstruction_gap))),
       "mean_atom_r2_in_sample": float(metrics["atom_r2_in_sample"].mean())})
np.testing.assert_allclose(reconstruction_gap, 0.0, atol=1e-10)
if len(metrics) != len(mmp_rows):
    raise ValueError("The canonical all-MMP population was not fully reconstructed")

output = ROOT / "notebooks_for_sglt2/output/reconstruction/csv"
output.mkdir(parents=True, exist_ok=True)
for filename, table in {"coverage": all_fit.audit, "metrics": all_fit.fit_metrics,
                        "coefficients": all_fit.coefficients, "exact_effects": all_fit.concept_deltas}.items():
    table.to_csv(output / f"{filename}.csv", index=False)


## 5. Inspect reconstruction fidelity

In [ ]:
from xai_chem.plotting import plot_reconstruction_yy, save_publication_figure
figure, axis = plt.subplots(figsize=(8, 5))
axis.hist(metrics["atom_r2_in_sample"], bins=40, color="steelblue")
axis.set(xlabel="In-sample atom regression R²", ylabel="MMP count")
save_publication_figure(figure, output.parent / "vis" / "atom_reconstruction_r2.png")
display(figure); plt.close(figure)
figure, scores = plot_reconstruction_yy(
    metrics["teacher_delta_B_minus_A"], metrics["fitted_delta_B_minus_A"],
    unit_label="B−A fragment contribution (pIC50)", title="In-sample MMP reconstruction",
)
save_publication_figure(figure, output.parent / "vis" / "delta_reconstruction_yy.png")
display(figure); plt.close(figure)
display(scores)


The fidelity plot measures reconstruction of a fixed teacher on the fitted atoms. It does not measure held-out predictive accuracy or establish causal descriptor effects. MMPs share atoms and molecules, so the 23,029 fits are not independent replicates. The original regularization, pair orientation, atom mapping, and raw physical descriptor values are preserved.